# Build a bifurcation diagram
Run the cell, then press Play. Each parameter starts a fresh trajectory at x₀ = 0.1; discard 12,000 updates and plot the next 256 values.
Use the slider to move backward or forward. The lower plot shows the current trajectory after settling.

In [ ]:
from IPython.display import HTML, display

# Self-contained animation: no ipywidgets or external libraries needed.
animation_html = '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>From one rule to many patterns</title>\n<style>\nbody{margin:0;background:#f4f6fa;color:#172a3a;font:16px system-ui,sans-serif}main{max-width:1180px;margin:auto;padding:28px}h1{font-size:28px;margin:0 0 8px}p{color:#526477;line-height:1.5}.rule{font-size:20px;color:#176b82}.card{background:white;border:1px solid #dce3ec;border-radius:16px;padding:18px;margin-top:18px}.top{display:flex;justify-content:space-between;gap:15px;align-items:center}.badge{background:#e6f5f4;color:#087b77;padding:8px 14px;border-radius:20px;font-weight:650}canvas{width:100%;display:block}#plot{height:420px}#series{height:160px}.controls{display:flex;flex-wrap:wrap;align-items:center;gap:14px}button,select{font:inherit;border:1px solid #c7d4df;background:white;color:#172a3a;border-radius:8px;padding:9px 14px;cursor:pointer}#play{background:#087f83;color:white;border-color:#087f83}input{accent-color:#087f83;flex:1;min-width:150px}small{color:#526477}.examples button{padding:5px 10px;font-size:14px}.examples{margin-top:12px;display:flex;flex-wrap:wrap;gap:8px}.legend{color:#526477;font-size:14px}#status{min-height:24px}button:disabled{opacity:.5}\n</style></head><body><main>\n<h1>From one rule to many patterns</h1>\n<div class="rule">x<sub>t+1</sub> = r x<sub>t</sub>(1 − x<sub>t</sub>)</div>\n<p>Turn one parameter. Let each trajectory settle. Keep its late values—and watch the diagram grow.</p>\n<div class="card"><div class="top"><strong>Bifurcation diagram</strong><span class="badge" id="reading">Preparing…</span></div>\n<canvas id="plot" aria-label="Bifurcation diagram showing r horizontally and late trajectory values vertically"></canvas>\n<div class="legend">Teal: values collected so far · Orange: current parameter · Gray line: where the sweep is now</div></div>\n<div class="card"><div class="top"><strong>The current trajectory, after settling</strong><span id="period"></span></div>\n<canvas id="series" aria-label="Late time series for the current parameter"></canvas>\n<small>Each column comes from a fresh run at x₀ = 0.1. The lower plot shows 60 consecutive late values.</small></div>\n<div class="card"><div class="controls">\n<button id="play" disabled>▶ Play</button><button id="reset" disabled>Reset</button>\n<label for="r">r</label><input id="r" type="range" min="0" max="1800" value="0" step="1" disabled>\n<label for="speed">Sweep time</label><select id="speed"><option value="60">60 s</option><option value="30" selected>30 s</option><option value="15">15 s</option></select>\n</div><div class="examples"><button data-r="2.5">2.50 · fixed point</button><button data-r="3.2">3.20 · period 2</button><button data-r="3.5">3.50 · period 4</button><button data-r="3.55">3.55 · period 8</button><button data-r="3.7">3.70 · irregular</button><button data-r="3.83">3.83 · period 3 returns</button><button data-r="4">Full diagram</button></div>\n<p id="status" role="status">Preparing the trajectories…</p>\n<small>The vertical axis shows points visited by the trajectory, not the number of attractors. Period labels are numerical estimates. No detected short period does not establish chaos. Finite settling time can leave transients near bifurcations.</small>\n</div></main><script>\n(() => {\nconst N=1801, KEEP=256, BURN=12000, RMIN=2.5, RMAX=4;\nconst $=id=>document.getElementById(id), plot=$(\'plot\'), series=$(\'series\');\nconst data=new Float64Array(N*KEEP), periods=new Uint8Array(N);\nconst layer=document.createElement(\'canvas\'); let ready=false,index=0,painted=-1,playing=false,last=0,progress=0;\nconst rAt=i=>RMIN+(RMAX-RMIN)*i/(N-1);\nfunction periodAt(i){\n const start=i*KEEP,r=rAt(i);\n for(let p=1;p<=16;p++){\n  let ok=true;for(let j=p;j<KEEP;j++)if(Math.abs(data[start+j]-data[start+j-p])>1e-10){ok=false;break;}\n  if(!ok)continue;\n  for(const end of [128,256]){let m=1;for(let j=end-p;j<end;j++)m*=r*(1-2*data[start+j]);if(Math.abs(m)>=1-1e-6){ok=false;break;}}\n  if(ok)return p;\n }return 0;\n}\nfunction setup(canvas){const d=window.devicePixelRatio||1,w=canvas.clientWidth,h=canvas.clientHeight;canvas.width=Math.round(w*d);canvas.height=Math.round(h*d);const c=canvas.getContext(\'2d\');c.setTransform(d,0,0,d,0,0);return {c,w,h};}\nfunction axes(c,w,h,isSeries=false){\n const box={l:58,t:18,w:w-78,h:h-58};c.clearRect(0,0,w,h);c.font=\'12px system-ui\';c.lineWidth=1;\n for(let k=0;k<=4;k++){const y=box.t+box.h*(1-k/4);c.strokeStyle=\'#e8edf3\';c.beginPath();c.moveTo(box.l,y);c.lineTo(box.l+box.w,y);c.stroke();c.fillStyle=\'#526477\';c.textAlign=\'right\';c.fillText((k/4).toFixed(2),box.l-9,y+4);}\n c.textAlign=\'center\';for(let k=0;k<=6;k++){const x=box.l+box.w*k/6;c.fillText(isSeries?String(k*10): (RMIN+1.5*k/6).toFixed(2),x,h-21);}\n c.fillText(isSeries?\'Late observation (after settling)\':\'Parameter r\',box.l+box.w/2,h-3);\n c.save();c.translate(14,box.t+box.h/2);c.rotate(-Math.PI/2);c.fillText(isSeries?\'xₜ\':\'Late values of x\',0,0);c.restore();return box;\n}\nfunction draw(){\n if(!ready)return;\n const {c,w,h}=setup(plot),b=axes(c,w,h),d=window.devicePixelRatio||1;\n if(layer.width!==plot.width||layer.height!==plot.height){layer.width=plot.width;layer.height=plot.height;painted=-1;}\n const lc=layer.getContext(\'2d\');lc.setTransform(d,0,0,d,0,0);\n if(index<painted){lc.clearRect(0,0,w,h);painted=-1;}\n lc.fillStyle=\'rgba(8,115,127,.18)\';\n for(let i=painted+1;i<=index;i++){const x=b.l+b.w*i/(N-1);for(let j=0;j<KEEP;j++){const y=b.t+b.h*(1-data[i*KEEP+j]);lc.fillRect(x,y,1.1,1.1);}}\n painted=index;c.drawImage(layer,0,0,w,h);\n const x=b.l+b.w*index/(N-1);c.strokeStyle=\'#9aa9b5\';c.beginPath();c.moveTo(x,b.t);c.lineTo(x,b.t+b.h);c.stroke();\n c.fillStyle=\'rgba(234,112,39,.7)\';for(let j=0;j<KEEP;j++){c.beginPath();c.arc(x,b.t+b.h*(1-data[index*KEEP+j]),2,0,Math.PI*2);c.fill();}\n const s=setup(series),sb=axes(s.c,s.w,s.h,true);s.c.strokeStyle=\'#e77729\';s.c.lineWidth=1.5;s.c.beginPath();\n for(let j=0;j<60;j++){const xx=sb.l+sb.w*j/60,yy=sb.t+sb.h*(1-data[index*KEEP+KEEP-60+j]);j?s.c.lineTo(xx,yy):s.c.moveTo(xx,yy);}s.c.stroke();\n s.c.fillStyle=\'#087f83\';for(let j=0;j<60;j++){s.c.beginPath();s.c.arc(sb.l+sb.w*j/60,sb.t+sb.h*(1-data[index*KEEP+KEEP-60+j]),2.4,0,Math.PI*2);s.c.fill();}\n $(\'reading\').textContent=\'r = \'+rAt(index).toFixed(4);$(\'r\').value=index;\n $(\'period\').textContent=periods[index]?\'Estimated period: \'+periods[index]:\'No stable period ≤16 confirmed\';\n $(\'status\').textContent=index===N-1?\'Sweep complete. Notice the periodic windows inside the irregular region.\':\'Pause anywhere: one column contains many late observations at a single r.\';\n}\nfunction stop(){playing=false;$(\'play\').textContent=\'▶ Play\';}\nfunction tick(t){if(!playing)return;if(last)progress+=(t-last)/1000*(N-1)/Number($(\'speed\').value);last=t;const next=Math.min(N-1,Math.floor(progress));if(next!==index){index=next;draw();}if(index===N-1)stop();else requestAnimationFrame(tick);}\nfunction jump(i){if(!ready)return;stop();index=i;progress=i;draw();}\n $(\'play\').onclick=()=>{if(playing){stop();return;}if(index===N-1){index=0;draw();}progress=index;last=0;playing=true;$(\'play\').textContent=\'Ⅱ Pause\';requestAnimationFrame(tick);};\n $(\'reset\').onclick=()=>jump(0);$(\'r\').oninput=()=>jump(Number($(\'r\').value));\n document.querySelectorAll(\'[data-r]\').forEach(button=>button.onclick=()=>jump(Math.round((Number(button.dataset.r)-RMIN)/(RMAX-RMIN)*(N-1))));\n new ResizeObserver(()=>{painted=-1;layer.width=0;draw();}).observe(plot);\n async function prepare(){for(let i=0;i<N;i++){let x=.1,r=rAt(i);for(let t=0;t<BURN;t++)x=r*x*(1-x);for(let j=0;j<KEEP;j++){x=r*x*(1-x);data[i*KEEP+j]=x;}periods[i]=periodAt(i);if(i%60===0){$(\'reading\').textContent=\'Preparing \'+Math.round(100*i/N)+\'%\';await new Promise(resolve=>setTimeout(resolve,0));}}ready=true;for(const id of [\'play\',\'reset\',\'r\'])$(id).disabled=false;draw();}prepare();\n})();\n</script></body></html>\n'
display(HTML(animation_html))
